# Lesson 02 - Microsoft Agent Framework ਦੀ ਖੋਜ

**Microsoft Agent Framework (MAF)** ਏਆਈ ਏਜੰਟ ਬਣਾਉਣ ਲਈ ਇੱਕ ਇਕੱਠਾ ਫਰੇਮਵਰਕ ਹੈ। ਇਹ ਚਾਰ ਮੁੱਖ ਬਿਲਡਿੰਗ ਬਲਾਕਾਂ ਨਾਲ ਇੱਕ ਸਾਫ਼, ਸੰਯੋਜਿਤ ਆਰਕੀਟੈਕਚਰ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ:

- **Client** – ਏਆਈ ਮਾਡਲ ਅੰਤਬਿੰਦੂ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਸੰਚਾਰ ਨੂੰ ਸੰਭਾਲਦਾ ਹੈ
- **Agent** – ਨਿਰਦੇਸ਼ਾਂ ਅਤੇ ਸੰਦ ਪਰਿਭਾਸ਼ਾਵਾਂ ਨਾਲ ਇੱਕ ਕਲਾਇੰਟ ਨੂੰ ਲਪੇਟਦਾ ਹੈ
- **Tools** – ਮਾਡਲ ਦੁਆਰਾ ਕਾਲ ਕੀਤੇ ਜਾ ਸਕਣ ਵਾਲੇ ਕਸਟਮ ਫੰਕਸ਼ਨਾਂ ਨਾਲ ਏਜੰਟ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਵਧਾਉਂਦੇ ਹਨ
- **Session** – ਬਹੁ-ਮੋੜ ਵਾਲੀਆਂ ਮੁਲਾਕਾਤਾਂ ਲਈ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਬਣਾਈ ਰੱਖਦਾ ਹੈ

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਇੱਕ **ਯਾਤਰਾ ਬੁਕਿੰਗ ਏਜੰਟ** ਬਣਾਵਾਂਗੇ ਜੋ ਇਹ ਸੰਕਲਪ ਵਰਤ ਕੇ ਗੰਤव्य ਉਪਲਬਧਤਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਆਰਕੀਟੈਕਚਰ ਨੂੰ ਸਮਝਣਾ

Microsoft Agent Framework ਇੱਕ ਲੇਅਰਡ ਆਰਕੀਟੈਕਚਰ ਨੂੰ ਫਾਲੋ ਕਰਦਾ ਹੈ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ਕਲਾਇੰਟ** – ਇੱਕ `AzureAIProjectAgentProvider` Azure OpenAI ਡਿਪਲੋਇਮੈਂਟ ਨਾਲ ਜੁੜਦਾ ਹੈ। ਇਹ ਪ੍ਰਮਾਣਿਕਤਾ, ਬੇਨਤੀ ਫਾਰਮੈਟਿੰਗ ਅਤੇ ਜਵਾਬ ਪਾਰਸ ਕਰਨਾ ਸੰਭਾਲਦਾ ਹੈ।
2. **ਏਜੰਟ** – ਕਲਾਇੰਟ ਤੋਂ `provider.create_agent()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ, ਏਜੰਟ ਮਾਡਲ ਐਕਸੈਸ ਨੂੰ ਨਿਰਦੇਸ਼ਾਂ (ਸਿਸਟਮ ਪ੍ਰੰਪਟ) ਅਤੇ ਟੂਲਾਂ ਨਾਲ ਮਿਲਾਉਂਦਾ ਹੈ।
3. **ਟੂਲ** – ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਜੋ `@tool` ਨਾਲ ਆਲੰਕ੍ਰਿਤ ਹੁੰਦੇ ਹਨ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਾਰਵਾਈ ਕਰਨ ਜਾਂ ਡੇਟਾ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
4. **ਸੈਸ਼ਨ** – ਇੱਕ `AgentSession` ਵਸਤੂ (ਜੋ `agent.create_session()` ਰਾਹੀਂ ਬਣਾਈ ਜਾਂਦੀ ਹੈ) ਜੋ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਸੰਗ੍ਰਹਿਤ ਕਰਦੀ ਹੈ, ਅਜਿਹੀ ਕਿ ਏਜੰਟ ਪਹਿਲਾਂ ਦੇ ਸੰਦਰਭ ਨੂੰ ਯਾਦ ਰੱਖਦਾ ਹੋਇਆ ਬਹੁ-ਚਰੰ ਮੁਲਾਕਾਤ ਕਰ ਸਕੇ।

ਆਓ ਹਰ ਲੇਅਰ ਨੂੰ ਕਦਮ ਦਰ ਕਦਮ ਬਣਾਈਏ।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ਡੈਕੋਰੇਟਰ ਨਾਲ ਟੂਲ ਸ਼ਾਮਲ ਕਰਨਾ

ਟੂਲ ਏਜੰਟਾਂ ਨੂੰ ਟੈਕਸਟ ਬਣਾਉਣ ਤੋਂ ਬਾਹਰ ਕਾਰਵਾਈਆਂ ਕਰਨ ਦਿੰਦੇ ਹਨ। `@tool` ਡੈਕੋਰੇਟਰ ਇੱਕ ਆਮ ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਨੂੰ ਕੁਝ ਐਸਾ ਬਣਾ ਦਿੰਦਾ ਹੈ ਜੋ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਮੁੱਖ ਬਿੰਦੂ:
- ਮਾਡਲ ਨੂੰ ਹਰ ਪੈਰਾਮੀਟਰ ਸਮਝ ਆ ਸਕੇ ਇਸ ਲਈ `Annotated[type, "description"]` ਵਰਤੋ।
- ਡੌਕਸਟਰਿੰਗ ਟੂਲ ਦਾ ਵੇਰਵਾ ਬਣ ਜਾਂਦੀ ਹੈ ਜੋ ਮਾਡਲ ਵੇਖਦਾ ਹੈ।
- `approval_mode="never_require"` ਦਾ ਮਤਲਬ ਹੈ ਕਿ ਟੂਲ ਬਿਨਾਂ ਯੂਜ਼ਰ ਦੀ ਪੁਸ਼ਟੀ ਦੇ ਆਪਣੇ ਆਪ ਚਲਦਾ ਹੈ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ਟੂਲਾਂ ਨਾਲ ਇੱਕ ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਕਲਾਇੰਟ, ਹੁਕਮਾਂ, ਅਤੇ ਟੂਲਾਂ ਨੂੰ ਇਕੱਠੇ ਕਰਕੇ ਇੱਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ। `instructions` ਸਿਸਟਮ ਪ੍ਰਾਰੰਭਿਕ ਹੁਕਮ ਵਜੋਂ ਕੰਮ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਦੀ ਪਹਚਾਣ ਅਤੇ ਵਰਤਾਰਾ ਨਿਰਧਾਰਤ ਕਰਦੇ ਹਨ।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ਸੈਸ਼ਨਾਂ ਨਾਲ ਮੁਲਟੀ-ਟਰਨ ਗੱਲਬਾਤਾਂ

ਇੱਕ `AgentSession` (`agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਗੱਲਬਾਤ ਵਿੱਚ ਸਾਰੀਆਂ ਸੰਦੇਸ਼ਾਂ ਦਾ ਟ੍ਰੈਕ ਰੱਖਦਾ ਹੈ। ਇੱਕੋ ਸੈਸ਼ਨ ਨੂੰ ਹਰ `agent.run()` ਕਾਲ ਨਾਲ ਪਾਸ ਕਰਕੇ, ਏਜੰਟ ਨੂੰ ਪੂਰੀ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਤੱਕ ਪਹੁੰਚ ਮਿਲਦੀ ਹੈ ਅਤੇ ਉਹ ਪਹਿਲਾਂ ਦੇ ਸੰਦੇਸ਼ਾਂ ਨੂੰ ਵਾਪਸ ਜਾ ਕੇ ਵੇਖ ਸਕਦਾ ਹੈ।

ਅਸੀਂ `tools=[check_destination_availability]` ਪਾਸ ਕਰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਏਜੰਟ ਹਰ ਟਰਨ ਦੌਰਾਨ ਸਾਡੇ ਉਪਲਬਧਤਾ ਜਾਂਚਕਨ ਵਾਲੇ ਨੂੰ ਕਾਲ ਕਰ ਸਕੇ।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੈਂਟ ਫ੍ਰੇਮਵਰਕ ਦੇ ਚਾਰ ਕਾਲਮਾਂ ਦੀ ਖੋਜ ਕੀਤੀ:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` ਸਾਕਸ਼ਰਤਾ-ਅਧਾਰਤ ਪ੍ਰਮਾਣਿਕਤਾ ਨਾਲ Azure OpenAI ਨਾਲ ਕਨੈਕਟ ਹੁੰਦਾ ਹੈ |
| **Agent** | `provider.create_agent()` ਨਿਰਦੇਸ਼ਾਂ ਅਤੇ ਇੱਕ ਨਾਮ ਨਾਲ ਮਾਡਲ ਕਨੈਕਸ਼ਨ ਨੂੰ ਬੰਡਲ ਕਰਦਾ ਹੈ |
| **Tools** | `@tool` ਡੈਕੋਰੇਟਰ ਏਜੈਂਟ ਲਈ ਕਾਲ ਕਰਨ ਲਈ ਪਾਇਥਨ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਪ੍ਰਗਟ ਕਰਦਾ ਹੈ |
| **Session** | `agent.create_session()` ਬਹੁਤ ਸਾਰੇ ਚਰਣਾਂ ਵਿੱਚ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਬਣਾਈ ਰੱਖਦਾ ਹੈ |

ਇਹ ਬਿਲਡਿੰਗ ਬਲਾਕ ਮਿਲ ਕੇ ਐਜੇੰਟ ਬਣਾਉਂਦੇ ਹਨ ਜੋ ਕੁਦਰਤੀ ਗੱਲਬਾਤ ਕਰ ਸਕਦੇ ਹਨ, ਬਾਹਰੀ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਕਾਲ ਕਰ ਸਕਦੇ ਹਨ, ਅਤੇ ਸੰਦਰਭ ਨੂੰ ਬਰਕਰਾਰ ਰੱਖਦੇ ਹਨ — ਇਹ ਅਗਲੇ ਪਾਠਾਂ ਵਿੱਚ ਜ਼ਿਆਦਾ ਅਧੁਨਿਕ ਏਜੈਂਟਿਕ ਪੈਟਰਨਾਂ ਲਈ ਨੀਂਹ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪੱਤਰ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਨਾਲ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਅਤ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੇ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਸਾਡੇ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਵਜੋਂ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਆਵਸ਼ਯਕ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ਾਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆ ਲਈ ਅਸੀਂ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
